**Human Movement Intelligence | Gait Analysis | Presented by: Reza Saadatyar (2026)**

**03 | Stride Measurement and Statistics**

![Stride Measurement and Statistics](../../images/03_stride_measurement_and_statistics.svg)

**Objective:** Derive descriptive stride timing from independent force-contact references without using reserved Test outcomes for model development.

**Dataset:** Camargo et al. (2021), source-validated original-time paired IMU and independent force-derived contact references.

**Notebook scope:** Development/Test boundaries and stated measurement units are preserved. Numerical results are only established after original-data execution.


**Current pilot pathway:** Source-phase annotation agreement, not independently validated GRF event timing. The original force-reference pathway is preserved separately; selected runs must not conflate these two evidence types.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
# This IPython-native checkout requires no imports outside Section 01.
repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = "/content/human-movement-intelligence"
shell = get_ipython()
if shell is None:
    raise RuntimeError("Execute this notebook in Google Colab or IPython.")
shell.system(
    f'if [ ! -d "{repo_path}" ]; then git clone -q "{repo_url}" '
    f'"{repo_path}"; fi'
)

# ---------------------------- 2. Open Repository -----------------------------
shell.run_line_magic("cd", repo_path)


**01. Imports and Research Environment**

Read validated Notebook 02 reference events, eligible trials and participant roles. Missing inputs raise an actionable dependency error; descriptive source-contract outputs are insufficient. All subsequent reference summaries use Development participants only, without analysing held-out Test events.


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("gait_analysis")
NOTEBOOK_NUMBER = 3

# ----------------- 2. Scientific Computing and Data Analysis -----------------
from pathlib import Path
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 4. Study Research Utilities ------------------------
from gait_reference import reference_strides, stride_summary

# ---------------------- 5. Source and Output Locations -----------------------
study_dir = root / "gait_analysis"
output_dir = study_dir / "output"
# ------------------------ 6. Check Required Stage Inputs ---------------------
ANALYSIS_MODE = "source_phase_annotation"  # Exploratory pilot; not force truth
if ANALYSIS_MODE == "source_phase_annotation":
    required = [
        output_dir / "02_phase_annotation_events.csv",
        output_dir / "02_phase_annotation_trials.csv",
        output_dir / "02_phase_participant_roles.csv",
    ]
elif ANALYSIS_MODE == "validated_force":
    required = [
        output_dir / "02_reference_events.csv",
        output_dir / "02_eligible_trials.csv",
        output_dir / "02_participant_roles.csv",
    ]
else:
    raise ValueError("Unknown gait study analysis mode.")
prefix = "03_phase" if ANALYSIS_MODE == "source_phase_annotation" else "03_reference"
print("Analysis mode:", ANALYSIS_MODE)
if ANALYSIS_MODE == "source_phase_annotation":
    print("Target: original gcRight phase annotations; not verified force IC.")

missing = [item.name for item in required if not item.is_file()]
if missing:
    raise FileNotFoundError(
        "Run the corresponding Notebook 02 preparation section. Missing: "
        + ", ".join(missing)
    )
references = pd.read_csv(
    required[0], dtype={"subject_id": str, "trial_key": str}
)
if ANALYSIS_MODE == "source_phase_annotation":
    if references.empty or not references.independent_force_reference_verified.eq(False).all():
        raise ValueError("Source-phase annotations are not verified force events.")
    references = references.rename(columns={"t_annotation_s": "t_reference_s"})
    references["event_type"] = "IC"
    references["t_reference_s"] = pd.to_numeric(
        references.t_reference_s, errors="raise"
    )

eligible = pd.read_csv(
    required[1], dtype={"subject_id": str, "trial_key": str}
)
roles = pd.read_csv(required[2], dtype={"subject_id": str})
if (roles.empty or roles.subject_id.duplicated().any()
        or not roles.role.isin(["Development", "Test"]).all()
        or not (roles.role == "Development").any()):
    raise ValueError("Notebook 02 participant roles must be unique and valid.")
if eligible.empty or eligible.trial_key.duplicated().any():
    raise ValueError("Eligible trial keys are missing or duplicated.")
if not set(eligible.subject_id).issubset(set(roles.subject_id)):
    raise ValueError("Some eligible participants have no assigned role.")
dev = eligible.merge(
    roles.loc[roles.role == "Development", ["subject_id"]],
    on="subject_id", validate="many_to_one"
)
if dev.empty:
    raise ValueError("No Development reference trials.")
references = references.merge(
    dev[["subject_id", "trial_key"]],
    on=["subject_id", "trial_key"], validate="many_to_one"
)
if references.empty or set(dev.trial_key) - set(references.trial_key):
    raise ValueError("Development trials missing force-reference events.")
print("Development participants:", dev.subject_id.nunique())
print("Reserved Test event times excluded from Development analysis.")
if ANALYSIS_MODE == "source_phase_annotation":
    print("Source annotations are an exploratory agreement target only.")


**02. Compute Per-Trial Reference Stride Measures**

Calculate consecutive same-foot Initial Contact intervals from independent force-derived events. Stride metrics require at least two successive complete contacts; inadequate trials remain visible in support counts.


In the exploratory pathway, all reported timing quantities describe original `gcRight` phase-annotation strides from Development participants, not calibrated force-reference contacts. The two reserved Test participants are excluded from this descriptive section.


In [ ]:
# ====================== 02. Calculate Reference Strides ======================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
rows, stride_rows = [], []
for trial in dev.itertuples(index=False):
    event = references.loc[references.trial_key == trial.trial_key]
    strides = reference_strides(event)
    summary = stride_summary(strides)
    rows.append({
        "subject_id": trial.subject_id, "session_id": trial.session_id,
        "condition": trial.condition, "trial_id": trial.trial_id,
        "trial_key": trial.trial_key, **summary
    })
    if len(strides):
        stride_rows.append(strides.assign(
            subject_id=trial.subject_id, trial_key=trial.trial_key,
            condition=trial.condition
        ))
trial_summary = pd.DataFrame(rows)
stride_table = (pd.concat(stride_rows, ignore_index=True)
                if stride_rows else pd.DataFrame(columns=[
                    "stride_index","start_s","end_s","stride_time_s",
                    "cadence_steps_per_min","subject_id","trial_key","condition"
                ]))

# ------------------------- 2. Save Scientific Tables -------------------------
trial_summary.to_csv(output_dir / f"{prefix}_trial_metrics.csv", index=False)
stride_table.to_csv(output_dir / f"{prefix}_strides.csv", index=False)
display(trial_summary.head())
print("Development trials:", len(trial_summary),
      "reference strides:", len(stride_table))


**03. Summarise Participants and Walking Conditions**

Summarise Development trials by participant and condition. This existing aggregation computes the unweighted mean of valid trial-level means and trial-level CVs; it is not a pooled within-person CV. Multiple trials or strides do not create additional independent participants.


In the exploratory pathway, all reported timing quantities describe original `gcRight` phase-annotation strides from Development participants, not calibrated force-reference contacts. The two reserved Test participants are excluded from this descriptive section.


In [ ]:
# ======================== 03. Summarise Participants =========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
good = trial_summary.loc[trial_summary.n_strides >= 2].copy()
if good.empty:
    raise ValueError("No adequate reference strides; review Notebook 02.")
subject_condition = (
    good.groupby(["subject_id", "condition"], as_index=False)
        .agg(n_trials=("trial_key", "nunique"),
             n_reference_strides=("n_strides", "sum"),
             mean_stride_time_s=("mean_stride_time_s", "mean"),
             stride_time_cv=("stride_time_cv", "mean"),
             cadence_steps_per_min=("cadence_steps_per_min", "mean"))
)

# ------------------------- 2. Save Scientific Tables -------------------------
subject_condition.to_csv(
    output_dir / ("03_phase_subject_condition_metrics.csv" if
                  ANALYSIS_MODE == "source_phase_annotation" else
                  "03_subject_condition_metrics.csv"), index=False
)
display(subject_condition.head())
print("Independent Development people:", subject_condition.subject_id.nunique())


**04. Visualise Stride Timing and Within-Person Variability**

Plot recorded Development measures only. The CV panel shows mean trial-level stride-time CV rather than a pooled within-person CV; condition labels reflect only eligible observations.


In the exploratory pathway, all reported timing quantities describe original `gcRight` phase-annotation strides from Development participants, not calibrated force-reference contacts. The two reserved Test participants are excluded from this descriptive section.


In [ ]:
# ====================== 04. Visualise Reference Strides ======================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
conditions = sorted(subject_condition.condition.unique())
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
for i, cond in enumerate(conditions):
    subset = subject_condition.loc[subject_condition.condition == cond]
    axes[0].scatter([i] * len(subset), subset.mean_stride_time_s,
                    alpha=0.7, label=cond)
    axes[1].scatter([i] * len(subset), subset.stride_time_cv,
                    alpha=0.7, label=cond)
for ax, title, unit in zip(axes,
    [("Source-phase stride time" if ANALYSIS_MODE == "source_phase_annotation"
      else "Mean force-derived stride time"),
     "Mean trial-level stride-time CV"],
    ["Stride time (s)","Coefficient of variation"]):
    ax.set_xticks(range(len(conditions)), conditions, rotation=15)
    ax.set(title=title, ylabel=unit)
    ax.grid(alpha=0.2)

# ------------------------ 2. Export Scientific Figure ------------------------
fig.tight_layout()
save_figure(fig, output_dir / f"{prefix}_stride_variability.png")
plt.show()


**05. Interpret Reference Descriptive Results**

Keep reference timing separate from detector performance. If only treadmill is eligible, no terrain-generalisation comparison can be claimed. The held-out Test people are not used in this descriptive Development summary.


In the exploratory pathway, all reported timing quantities describe original `gcRight` phase-annotation strides from Development participants, not calibrated force-reference contacts. The two reserved Test participants are excluded from this descriptive section.


In [ ]:
# ====================== 05. Interpret Reference Results ======================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
print("Observed conditions:", conditions)
print("Trials without two complete strides:",
      int((trial_summary.n_strides < 2).sum()))
print("Notebook 04 must compare IMU events with these "
      "source-annotation timestamps on Development folds.")
if ANALYSIS_MODE == "source_phase_annotation":
    print("No independently verified force-contact error is claimed.")


**06. Research Summary and Output Download**

The saved `03_` results describe Development-only force-reference stride timing, cadence and variability. These are reference measurements, not estimates of IMU model accuracy. Conclusions depend on the participant and condition coverage in the executed tables.

The shared helper packages the saved `03_` research tables and figures into one ZIP **outside the repository**. Retrieve it from Colab's file browser. Raw Camargo archives are not included, and no output is pushed to GitHub automatically.


In [ ]:
# ================= 06. Research Summary and Output Download ==================

# ------------------- 1. Review Current Scientific Outputs --------------------
print('Reference stride evidence saved to:', output_dir)

# ------------------- 2. Package Notebook-Specific Evidence -------------------
download_outputs_from_colab(
    folder="gait_analysis/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
